In [11]:
import polars as pl

units = pl.read_parquet(
    "s3://aind-scratch-data/dynamic-routing/cache/nwb_components/v0.0.272/consolidated/units.parquet"
)


In [ ]:
structure_grouping = {
    'SCop': 'SCs',
    'SCsg': 'SCs',
    'SCzo': 'SCs',
    'SCig': 'SCm',
    'SCiw': 'SCm',
    'SCdg': 'SCm',
    'SCdw': 'SCm',
    "ECT1": 'ECT',
    "ECT2/3": 'ECT',    
    "ECT6b": 'ECT',
    "ECT5": 'ECT',
    "ECT6a": 'ECT', 
    "ECT4": 'ECT',
}
units.with_columns(pl.col('structure').replace(structure_grouping))

structure
str
"""SCm"""
"""SCm"""
"""SCm"""
"""SCm"""
"""SCm"""
…
"""ECT"""
"""ECT"""
"""ECT"""


In [ ]:
import polars as pl

(
    pl.scan_parquet(
        "s3://aind-scratch-data/dynamic-routing/cache/nwb_components/v0.0.272/consolidated/units.parquet"
    )
    .join(
        pl.scan_parquet(
            "s3://aind-scratch-data/dynamic-routing/cache/nwb_components/v0.0.272/consolidated/session.parquet"
        ).filter(
            pl.col('keywords').list.contains('ccf'),
            pl.col('keywords').list.contains('task'),
            pl.col('keywords').list.contains('production'),
            # ~pl.col('keywords').list.contains('issues'),
            ~pl.col('keywords').list.contains('templeton'),
        ),
        on=['session_id'],
        how='semi',
    )
    .with_columns(
        (
            
        pl.col('activity_drift').le(0.2) & pl.col('amplitude_cutoff').le(0.1)
        & pl.col('presence_ratio').ge(0.7)
        & pl.col('isi_violations_ratio').le(0.5)
        & pl.col('decoder_label').ne('noise')
        # & pl.col('firing_rate').ge(1)
        ).alias('default_qc')
    )
    .group_by('session_id')
    .agg(
        pl.col('unit_id').filter('default_qc').n_unique().alias('num_good_units'),
        pl.col('unit_id').filter(~pl.col('default_qc')).n_unique().alias('num_bad_units'),
    )
    .sort('num_good_units')
    .collect()
    ['num_good_units'].sum()
)

173228